In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import time

# Load frozen embedding table
embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 16384
MiniLM dim : 384


In [2]:
512/4, 512-384

(128.0, 128)

In [4]:
import torch.nn.functional as F 

DIM      = 384
EXPANDED_DIM = 256
N_HEADS  = 8
N_LAYERS = 24
FFN_DIM  = 512
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)


class MicroLM(nn.Module):
    """
    MicroLM using PyTorch built-in TransformerEncoderLayer.

    Flow:
      token_ids
        -> frozen embedding_table lookup
        -> expander: 384 -> 512
        -> sinusoidal positional encoding
        -> N transformer encoder layers with causal mask
        -> final LayerNorm
        -> output_head
    """

    def __init__(self):
        super().__init__()

        # Frozen MiniLM-style embedding table
        self.register_buffer("embedding_table", embedding_tensor)

        # Expander 384 -> 512
        self.expander = nn.Sequential(
            nn.Linear(MINILM_DIM, EXPANDED_DIM),
            nn.LayerNorm(EXPANDED_DIM),
        )

        # Built-in transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EXPANDED_DIM,
            nhead=N_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=0.1,
            activation="gelu",
            batch_first=True,     # input/output shape: (B, T, D)
            norm_first=True       # pre-norm, same style as your original block
            
        )

        self.blocks = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=N_LAYERS
        )

        self.norm = nn.LayerNorm(EXPANDED_DIM)
        self.output_head = nn.Linear(EXPANDED_DIM, VOCAB_SIZE, bias=False)

    def make_causal_mask(self, T, device):
        """
        PyTorch Transformer expects mask shape: (T, T)

        True means blocked when using bool mask.
        So upper triangle = True.
        """
        return torch.triu(
            torch.ones(T, T, device=device, dtype=torch.bool),
            diagonal=1
        )

    def forward(self, token_ids):
        B, T = token_ids.shape

        # (B, T, 384)
        x = self.embedding_table[token_ids]

        # (B, T, 512)
        x = self.expander(x)

        # Add positional encoding
        x = x + sinusoidal_encoding(T, EXPANDED_DIM, token_ids.device)

        # Causal mask for GPT-style left-to-right generation
        causal_mask = self.make_causal_mask(T, token_ids.device)

        # (B, T, 512)
        x = self.blocks(x, mask=causal_mask)

        x = self.norm(x)

        # (B, T, VOCAB_SIZE)
        logits = self.output_head(x)

        return logits

In [5]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

# print(f"\nParam breakdown:")
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(f"  {name:55s} {p.numel():>10,}")

# Forward pass
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"\nInput  : {x.shape}")
print(f"Output : {logits.shape}")

/tmp/ipykernel_59713/1262110867.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


Trainable params : 16,944,384
Frozen params    : 0  (embedding table)
Total params     : 16,944,384

Input  : torch.Size([2, 32])
Output : torch.Size([2, 32, 16384])


In [6]:
model

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): TransformerEncoder(
    (layers): ModuleList(
      (0-23): 24 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (output_head): Linear(in_features=256, out_features=16384, bias=

In [7]:
total     = sum(p.numel() for p in model.expander.parameters())
trainable = sum(p.numel() for p in model.expander.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 99,072
Frozen params    : 0  (embedding table)
Total params     : 99,072


In [10]:
(2*268435456)/((16944384 - 99072) + 256 * 16000)

25.636928192464733

In [11]:
# ── Training Config ───────────────────────────────────────────────────────────
import gc
import json
import random
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

SHARD_DIR  = Path("dataset_shards_v2")
SEQ_LEN    = 512
BATCH_SIZE = 32
N_EPOCHS   = 3
LOG_EVERY  = 200
CKPT_DIR   = Path("checkpoints_v7")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Device      : cuda
Model params: 16,944,384


In [12]:
from torch.utils.data import Dataset

SEQ_LEN = 512    # window size
STRIDE  = 448    # step size → 64-token overlap

class ShardDataset(Dataset):
    def __init__(self, shard_path, seq_len=SEQ_LEN, stride=STRIDE):
        flat = torch.load(shard_path, weights_only=True)

        self.flat    = flat.to(torch.int16)
        self.seq_len = seq_len
        self.stride  = stride

        # number of full windows we can extract
        # +1 because we need seq_len+1 tokens (x + y target)
        n_tokens = len(self.flat)
        self.num_samples = max(0, (n_tokens - seq_len - 1) // stride + 1)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start  = idx * self.stride
        tokens = self.flat[start : start + self.seq_len + 1].to(torch.long)  # ← cast here
        
        if len(tokens) < self.seq_len + 1:
            pad    = torch.full((self.seq_len + 1 - len(tokens),), PAD_ID, dtype=torch.long)
            tokens = torch.cat([tokens, pad])

        x = tokens[:-1]
        y = tokens[1:]
        return x, y


BOS_ID       = 2
EOS_ID       = 3

def collate_fn(batch):
    xs, ys = zip(*batch)
    # all samples are now fixed-length (seq_len), so no padding needed here
    x_pad = torch.stack(xs)          # [B, seq_len]
    y_pad = torch.stack(ys).clone()  # [B, seq_len]

    # mask out targets after the first EOS in each sequence
    for i, y in enumerate(y_pad):
        eos_positions = (y == EOS_ID).nonzero(as_tuple=True)[0]
        if len(eos_positions) > 0:
            eos_pos = eos_positions[0].item()
            if eos_pos + 1 < y.size(0):
                y_pad[i, eos_pos + 1:] = -100

    return x_pad, y_pad

from tokenizers import Tokenizer
TOKENIZER_PATH   = "tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
dataset = ShardDataset("dataset_shards/shard_0000.pt")
for i in range(10,11):
    x, y = dataset[i]
    print(f"\nSample {i}")
    print(f"  x: {x.tolist()}")
    print(f"  y: {tokenizer.decode(x.tolist(), skip_special_tokens=False)}")
    # print(f"  y: {y.tolist()}")
for idx  in x.tolist():
    tok = tokenizer.decode([idx], skip_special_tokens=False)
    print(f"{idx} === '{tok}' ")


In [13]:
# run once after sharding
# meta = {
#     "total_samples": sum(len(ShardDataset(p)) for p in sorted(SHARD_DIR.glob("shard_*.pt"))),
#     "seq_len": SEQ_LEN,
#     "stride": STRIDE,
# }
# with open(SHARD_DIR / "meta.json", "w") as f:
#     json.dump(meta, f, indent=2)
    
# ── Optimizer & Scheduler ─────────────────────────────────────────────────────

with open(SHARD_DIR / "meta.json") as f:
    meta = json.load(f)

total_steps = (meta['total_samples'] // BATCH_SIZE) * N_EPOCHS

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-5
)
scaler    = torch.cuda.amp.GradScaler()

print(f"Total samples: {meta['total_samples']:,}")
print(f"Total steps  : {total_steps:,}")

Total samples: 3,601,677
Total steps  : 337,656


/tmp/ipykernel_59713/892036717.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


In [14]:
model.to(device)

MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): TransformerEncoder(
    (layers): ModuleList(
      (0-23): 24 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (output_head): Linear(in_features=256, out_features=16384, bias=

In [15]:
shard_files  = sorted(SHARD_DIR.glob("shard_*.pt"))[:2]
# shard_files.extend(sorted(SHARD_DIR.glob("ins*.pt"))[:1])
# shard_files.extend(sorted(SHARD_DIR.glob("shard_*.pt"))[1:2])
shard_files

[PosixPath('dataset_shards_v2/shard_0000.pt'),
 PosixPath('dataset_shards_v2/shard_0001.pt')]

In [16]:
# ── Training Loop with Resume ─────────────────────────────────────────────────
import math
loss_history = []
start_epoch  = 0
start_shard  = 0

# ── Resume from latest checkpoint ────────────────────────────────────────────
# Check mid-epoch checkpoints first, then epoch checkpoints
all_ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
mid_ckpts  = [c for c in all_ckpts if "shard" in c.name]
epoch_ckpts = [c for c in all_ckpts if "shard" not in c.name]

if mid_ckpts:
    latest = mid_ckpts[-1]
    print(f"Resuming from mid-epoch checkpoint {latest}...")
    ckpt = torch.load(latest, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    loss_history = ckpt.get("loss_history", [])
    start_epoch  = ckpt["epoch"] - 1        # epoch is 1-indexed, range is 0-indexed
    start_shard  = ckpt["shard"] + 1        # resume from next shard
    print(f"Resumed — epoch {start_epoch+1}, starting from shard {start_shard+1}")

elif epoch_ckpts:
    latest = epoch_ckpts[-1]
    print(f"Resuming from epoch checkpoint {latest}...")
    ckpt = torch.load(latest, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    loss_history = ckpt.get("loss_history", [])
    start_epoch  = ckpt["epoch"]            # full epoch completed
    start_shard  = 0                        # start from beginning of next epoch
    print(f"Resumed — starting from epoch {start_epoch+1}")

else:
    print("No checkpoint found — training from scratch")

model.to(device)
model.train()

for epoch in range(start_epoch, N_EPOCHS):
    epoch_loss  = 0.0
    epoch_steps = 0

    # no shuffle — sequential shard order
    shard_order = list(range(len(shard_files)))

    # if resuming mid-epoch, skip already completed shards
    if epoch == start_epoch and start_shard > 0:
        shard_order = shard_order[start_shard:]
        print(f"Skipping shards 0-{start_shard-1}, resuming from shard {start_shard}")

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print(f"{'='*60}")

    for shard_num, shard_idx in enumerate(shard_order):
        # just before the step loop:
        shard_start = time.time()
        interval_start = time.time()
        
        # actual shard number in epoch (accounts for skipped shards on resume)
        actual_shard_num = shard_idx  

        shard_path = shard_files[shard_idx]
        print(f"\n[Epoch {epoch+1}] Shard {actual_shard_num+1}/{len(shard_files)} — {shard_path.name}")

        ds     = ShardDataset(shard_path)
        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=0,
            pin_memory=True,
        )

        shard_loss  = 0.0
        shard_steps = 0

        for step, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(x)
                loss   = F.cross_entropy(
                    logits.view(-1, logits.size(-1)),
                    y.view(-1),
                    ignore_index=-100,
                )
                del logits  # free 268MB immediately before backward


            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            if not math.isnan(loss.item()):
                shard_loss  += loss.item()
                shard_steps += 1
                epoch_loss  += loss.item()
                epoch_steps += 1

            if step % LOG_EVERY == 0:
                avg = shard_loss / max(shard_steps, 1)
                ppl = math.exp(min(avg, 20))
                lr  = scheduler.get_last_lr()[0]
                elapsed_total = time.time() - shard_start
                interval_time = time.time() - interval_start
                print(f"  step {step:>5} | loss {loss.item():.4f} | avg {avg:.4f} | ppl {ppl:.2f} | lr {lr:.2e} | last {LOG_EVERY} steps: {interval_time:.1f}s | total: {elapsed_total:.0f}s")
                interval_start = time.time()  # reset for next interval

            del loss, x, y
        shard_avg = shard_loss / max(shard_steps, 1)
        shard_ppl = math.exp(min(shard_avg, 20))
        loss_history.append({
            "epoch":      epoch + 1,
            "shard":      actual_shard_num,
            "avg_loss":   shard_avg,
            "perplexity": shard_ppl,
        })
        print(f"  Shard done — avg loss: {shard_avg:.4f} | ppl {shard_ppl:.2f}")

        del ds, loader
        gc.collect()
        torch.cuda.empty_cache()

        # mid-epoch checkpoint every 5 shards
        if (actual_shard_num + 1) % 2 == 0:
            ckpt_path = CKPT_DIR / f"epoch_{epoch+1}_shard_{actual_shard_num}.pt"
            torch.save({
                "epoch":        epoch + 1,
                "shard":        actual_shard_num,
                "model":        model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "scheduler":    scheduler.state_dict(),
                "loss_history": loss_history,
            }, ckpt_path)
            print(f"Mid-epoch checkpoint saved → {ckpt_path}")

    # reset start_shard after first resumed epoch
    start_shard = 0

    epoch_avg = epoch_loss / max(epoch_steps, 1)
    epoch_ppl = math.exp(min(epoch_avg, 20))
    print(f"\nEpoch {epoch+1} complete — avg loss: {epoch_avg:.4f} | ppl {epoch_ppl:.2f}")

    ckpt_path = CKPT_DIR / f"epoch_{epoch+1}.pt"
    torch.save({
        "epoch":        epoch + 1,
        "shard":        -1,
        "model":        model.state_dict(),
        "optimizer":    optimizer.state_dict(),
        "scheduler":    scheduler.state_dict(),
        "loss_history": loss_history,
    }, ckpt_path)
    print(f"Epoch checkpoint saved → {ckpt_path}")

    with open(CKPT_DIR / "loss_history.json", "w") as f:
        json.dump(loss_history, f, indent=2)

print("\nTraining complete.")

No checkpoint found — training from scratch

Epoch 1/3

[Epoch 1] Shard 1/2 — shard_0000.pt
  step     0 | loss 9.9242 | avg 9.9242 | ppl 20417.83 | lr 3.00e-04 | last 200 steps: 3.5s | total: 3s
  step   200 | loss 4.3449 | avg 5.3978 | ppl 220.93 | lr 3.00e-04 | last 200 steps: 26.3s | total: 30s
  step   400 | loss 3.8967 | avg 4.7389 | ppl 114.31 | lr 3.00e-04 | last 200 steps: 26.5s | total: 56s
  step   600 | loss 3.6083 | avg 4.4106 | ppl 82.32 | lr 3.00e-04 | last 200 steps: 26.9s | total: 83s
  step   800 | loss 3.3034 | avg 4.1872 | ppl 65.84 | lr 3.00e-04 | last 200 steps: 26.8s | total: 110s
  step  1000 | loss 3.1192 | avg 4.0222 | ppl 55.82 | lr 3.00e-04 | last 200 steps: 26.8s | total: 137s
  step  1200 | loss 3.0429 | avg 3.8882 | ppl 48.82 | lr 3.00e-04 | last 200 steps: 26.8s | total: 164s
  step  1400 | loss 3.0221 | avg 3.7783 | ppl 43.74 | lr 3.00e-04 | last 200 steps: 26.8s | total: 190s
  step  1600 | loss 3.0120 | avg 3.6841 | ppl 39.81 | lr 3.00e-04 | last 200 

In [17]:
# ── Inference ─────────────────────────────────────────────────────────────────
from tokenizers import Tokenizer

tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

# Special token IDs
BOS_ID       = 2
EOS_ID       = 3
USER_ID      = 5
ASSISTANT_ID = 6

SPECIAL_IDS = set()  # 0-49, all special tokens
MAX_SEQ = 2048
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=1, mode="text"):
    ids = tok.encode(prompt).ids

    if mode == "text":
        ids = [BOS_ID] + ids
    elif mode == "chat":
        ids = [BOS_ID, USER_ID] + ids + [ASSISTANT_ID]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs   = torch.softmax(top_vals, dim=-1)
            next_id = top_idx[0][torch.multinomial(probs, 1)]
            x = torch.cat([x, next_id.view(1, 1)], dim=1)
            if next_id.item() == EOS_ID:
                break

    # manually filter special token IDs before decoding
    clean_ids = [t for t in x[0].tolist() if t not in SPECIAL_IDS]
    return tok.decode(clean_ids)

# ── Text generation prompts ───────────────────────────────────────────────────

text_prompts = [
    "The quick brown fox",
    "Once upon a time",
    "What should i do ? i love her.",
    "Tell me about a girl named lola and her pet elephant"
]

print("=" * 60)
print("TEXT GENERATION MODE")
print("=" * 60)
for prompt in text_prompts:
    print(f"\nPrompt : {prompt}")
    print(f"Output : {generate(prompt, mode='text')}")

TEXT GENERATION MODE

Prompt : The quick brown fox
Output : The quick brown fox was walking in the woods. He was looking for something to eat. Suddenly, he heard a voice calling him.
"Hello there!" said the voice.
The fox was surprised. He had never heard a voice before. He looked around and saw a little bird.
"Hello," said the bird.
The fox was surprised. He had never heard a bird talk before.
"What is your name?" asked the bird.
The fox thought for a moment. Then he said, "My name is Foxy. What's your name?"
The bird replied, "My name is Bluey. What's yours?"
The fox smiled.
"My name is Foxy. What's yours?"
"My name is Foxy," said the bird.
"My name is Bluey," said Foxy.
"My name is Bluey. What's your name?"
"My name is Bluey," said the bird.
"My name is

Prompt : Once upon a time
Output : Once upon a time, there was a little girl named Sue. Sue loved to play with her toys and eat yummy food. One day, Sue found a big, red apple on the ground. She picked it up and said, "Wow, this app